# ..... and DeepSeek API Calls, Token Handling, and Recreating the DeepSeek Web UI with Gradio

This notebook is an English, GitHub-friendly version of the original article. The Python code cells are kept unchanged, as requested.


## Overview

This article provides API examples for large language model (LLM) calls needed for Agentic AI, with a focus on OpenAI-compatible API usage for ..... and DeepSeek.

It also shows how to use Gradio to recreate a DeepSeek-like web interaction interface for chatbot and Agentic AI debugging.

The examples were tested in practice and were generated with the help of DeepSeek.


## 1. Account setup and API key preparation

Before using the APIs, register for a ..... account and a DeepSeek account, then obtain your API keys.

Both ..... and DeepSeek support Alipay. DeepSeek officially provides a fast API with relatively low cost and supports both reasoning and non-reasoning modes.

..... has recently offered free token credits for testing, and it also supports multiple mainstream domestic models, which can be very useful for building Agentic AI systems in the current technical stack.

The API calls shown in this notebook cost less than one yuan in total.


### DeepSeek

1. Register a DeepSeek account.
2. Open the official website and click the API platform.

   https://www.deepseek.com/

3. Click **API keys** on the left, then click **Create new API key**.

4. Save the key locally.

5. Click **Top up**, choose **CNY**, and pay with Alipay or WeChat Pay.


### .....

1. Register a ..... account.

   https://www........./

2. Click the red **Console** button in the upper-right corner.

3. Click **Fees** → **Overview** in the upper-right corner.

4. Click **Top up** → **Alipay**. Choose the amount according to your service needs.

   ..... has recently offered a free token promotion. Developers may receive 10 million free tokens for testing. If you join this promotion, this case can be completed without payment.

5. Return to the homepage and click the red **Console** button again.

6. Click **Service Navigation** → **Artificial Intelligence**.

   https://www........./ui/console/index.html#/notebook

7. Click **Model API** in the lower-left corner.

   https://www........./ui/llm/

8. Click **Model API** → **API Key** on the left.

   https://www........./ui/llm/apikeys

9. Click the red **Create API Key** button and save the key locally.

At this point, the DeepSeek and ..... model API keys are ready.


## 2. Create a `.env` file

Inside the Jupyter Notebook project folder, create a file named `.env`.

Open the file and write the API keys in the following format:

```text
DEEPSEEK_API_KEY=sk-......
....._API_KEY=sk-.......
```

Then you can begin editing the notebook.


## 3. Install the required libraries

Install OpenAI, OpenAI Agents, and python-dotenv.

```bash
pip install openai python-dotenv openai-agents
```

If you are using Anaconda, this process may be more complicated. In practice, `pip` is more stable, or you can create a new environment.


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)


## 4. Create OpenAI clients

This is equivalent to specifying the cloud endpoint parameters for the chosen provider.


### DeepSeek client

In [ ]:
# Initialize DeepSeek client
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)


### ..... client

In [ ]:
# Initialize ..... client
client = OpenAI(
    api_key=os.getenv("....._API_KEY"),
    base_url="https://api........./api/llm/v1"
)


## 5. Demonstrate a conversation

```python
# Conversation with system, user, and assistant roles
messages = [
    {"role": "system", "content": "You are insightful academian."},
    # Sets the behavior, persona, rules, or constraints for the assistant.
    {"role": "assistant", "content": "Hello! How are you?"},
    # Represents the model’s own replies in the conversation history.
    {"role": "user", "content": "What can you do for me?"}
]
```

Then send `messages` through the API and let the LLM respond.


### DeepSeek example

In [ ]:
try:
    response = client.chat.completions.create(
        model="deepseek-chat",   # or "deepseek-reasoner"
        messages=messages,
        temperature=0.7,
        max_tokens=512
    )
    print("DeepSeek reply:", response.choices[0].message.content)
except Exception as e:
    print("DeepSeek error:", e)


### ..... example

In [ ]:
try:
    response = client.chat.completions.create(
        model="DeepSeek-R1-0528",   # or "Qwen3-30B-A3B" etc.
        messages=messages,
        temperature=0.7,
        max_tokens=512
    )
    print("..... reply:", response.choices[0].message.content)
except Exception as e:
    print("..... error:", e)


This completes the example of calling LLMs through ..... and DeepSeek APIs. It forms the foundation of cloud-based Agentic AI systems.

Users may then build Agentic AI systems directly with Pydantic, or deploy more quickly with the OpenAI SDK.


## 6. Recreate a DeepSeek-style chat interface with Gradio

To make testing easier, install Gradio and use it to reproduce a DeepSeek-style web interaction interface. This helps prepare for chatbot and Agentic AI debugging.

```bash
pip install gradio
```


### Gradio chat with conversation history

In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import gradio as gr

load_dotenv(override=True)

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"
)

SYSTEM_PROMPT = "You are insightful academian."

def chat_with_deepseek(message: str, history: list) -> str:
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]

    # Handle different history formats
    if history and isinstance(history[0], dict):
        # Dict format: either OpenAI style or Gradio native
        if "role" in history[0] and "content" in history[0]:
            messages.extend(history)          # already perfect
        elif "user" in history[0] and "bot" in history[0]:
            for turn in history:
                messages.append({"role": "user", "content": turn["user"]})
                messages.append({"role": "assistant", "content": turn["bot"]})
        else:
            # Unknown dict format – fallback: take first two values
            for turn in history:
                vals = list(turn.values())
                if len(vals) >= 2:
                    messages.append({"role": "user", "content": vals[0]})
                    messages.append({"role": "assistant", "content": vals[1]})
    elif history:
        # Assume list of tuples/lists (user, assistant)
        for turn in history:
            messages.append({"role": "user", "content": turn[0]})
            messages.append({"role": "assistant", "content": turn[1]})

    messages.append({"role": "user", "content": message})

    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=messages,
            temperature=0.7,
            max_tokens=512
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {e}"

demo = gr.ChatInterface(
    fn=chat_with_deepseek,
    title="DeepSeek Chat with Full History",
    description="Your entire conversation history is sent to the DeepSeek model with every message.",
    examples=[["What can you do for me?"], ["Tell me something interesting about AI."]]
)

if __name__ == "__main__":
    demo.launch()


The interface can be accessed locally in your browser:

```text
http://127.0.0.1:7860/
```

As noted above, DeepSeek and ..... provide many different LLM models.


## 7. List the LLM models provided by DeepSeek

```python
# list_....._models.py
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Initialize ..... client (using the same OpenAI-compatible client)
client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com/v1"  # This URL points to DeepSeek, not .....
)

try:
    # List available models
    models = client.models.list()
    print("Available models on .....:")
    for model in models.data:
        print(f"  - {model.id}")
except Exception as e:
    print(f"Error listing models: {e}")
```


## 8. List the LLM models provided by .....

```python
# list_....._models.py
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv(override=True)

# Initialize ..... client
client = OpenAI(
    api_key=os.getenv("....._API_KEY"),
    base_url="https://api........./api/llm/v1"
)

try:
    # List available models
    models = client.models.list()
    print("Available models on .....:")
    for model in models.data:
        print(f"  - {model.id}")
except Exception as e:
    print(f"Error listing models: {e}")
```


This makes it easy for small businesses and individual developers to quickly call a variety of LLMs for secondary development.

## Contact

- Business and recruitment: `yucongcai_business@outlook.com`
- Research-related inquiries: `yucongcai_research@outlook.com`
